# 02. Candidate Retrieval with Qdrant Vector Index (Stage 1)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TheNarhawl/mind-dataset-recsys/blob/main/notebooks/02_candidate_retrieval_qdrant.ipynb)

This notebook covers:
1. Embedding article representations `[{category}] {subcategory}: {title}. {abstract}` with `BAAI/bge-small-en-v1.5`.
2. Dynamic User Representation with temporal exponential decay:
   $$\mathbf{u} = \frac{\sum_{i=1}^N \lambda^{N-i} \mathbf{v}_i}{\sum_{i=1}^N \lambda^{N-i}}, \quad \lambda = 0.85$$
3. Building an in-process / HNSW vector index in **Qdrant**.
4. Benchmarking retrieval performance: **Recall@50**, **Recall@100**, and latency SLA (< 15 ms).


In [ ]:
# Setup environment
!pip install -q sentence-transformers qdrant-client pandas numpy tqdm
!pip install -q -e .


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch

from mind_recsys.retrieval.embedder import NewsEmbedder, compute_user_vector
from mind_recsys.retrieval.qdrant_index import QdrantNewsIndex

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device}")

news_df = pd.read_parquet("./data/processed/train_news.parquet")
behaviors_df = pd.read_parquet("./data/processed/train_behaviors.parquet")

embedder = NewsEmbedder(model_name="BAAI/bge-small-en-v1.5", device=device)
print(f"Embedder initialized. Embedding dimension: {embedder.dimension}")


In [ ]:
# Compute embeddings for all news articles
texts = news_df['text_repr'].tolist()
news_ids = news_df['news_id'].tolist()

embeddings = embedder.encode(texts, batch_size=256, show_progress_bar=True, normalize_embeddings=True)
print(f"Encoded {len(embeddings)} articles to matrix of shape {embeddings.shape}")


In [ ]:
# Index into Qdrant HNSW collection
qdrant_index = QdrantNewsIndex(collection_name="mind_news_small", vector_size=embedder.dimension, location=":memory:")
qdrant_index.create_collection(recreate=True)

payloads = news_df[['category', 'subcategory', 'title', 'abstract']].to_dict(orient='records')
qdrant_index.index_news(news_ids=news_ids, embeddings=embeddings, payloads=payloads, batch_size=1000)
print("Articles successfully indexed in Qdrant!")


In [ ]:
# Stage 1 Candidate Retrieval Evaluation (Recall@50, Recall@100 & Latency)
emb_dict = dict(zip(news_ids, embeddings))

eval_queries = []
for _, row in behaviors_df.head(2000).iterrows():
    history = row['parsed_history']
    clicked_in_imp = {nid for nid, click in row['parsed_impressions'] if click == 1}
    if not clicked_in_imp:
        continue

    hist_vecs = [emb_dict[nid] for nid in history if nid in emb_dict]
    if hist_vecs:
        u_vec = compute_user_vector(np.array(hist_vecs), decay_lambda=0.85)
    else:
        u_vec = np.zeros(embedder.dimension, dtype=np.float32)

    eval_queries.append((row['impression_id'], u_vec, clicked_in_imp))

metrics = qdrant_index.evaluate_recall(eval_queries, top_k_list=[50, 100])
print("Candidate Retrieval (Stage 1) Benchmarks:")
for k, v in metrics.items():
    print(f"  {k}: {v}")
